# 00 — What is stored in `X` and `layers['counts']`?

Before any analysis we check, separately for RNA (GEX) and protein (ADT) columns,
which normalization the provided `X` went through. Every claim below is tested on
the data, not taken from documentation.

The matrices are streamed from disk in blocks of cells (`citeseq.matrix_stats`),
so the full ~90k x 14k matrix is never loaded.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import h5py
import numpy as np
import pandas as pd

from citeseq.paths import CITE_H5AD
from citeseq.matrix_stats import (read_df_column, read_df_index,
                                  scan_by_feature_type, per_cell_ratio)

pd.set_option("display.float_format", "{:.4g}".format)
with h5py.File(CITE_H5AD) as f:
    feature_types = np.asarray(read_df_column(f["var"], "feature_types")).astype(str)
    size_factors = read_df_column(f["obs"], "GEX_size_factors")
pd.Series(feature_types).value_counts()

GEX    13953
ADT      134
Name: count, dtype: int64

## 1. Value statistics per feature type

In [2]:
results = {}
for key in ["layers/counts", "X"]:
    summary, per_cell, dense = scan_by_feature_type(CITE_H5AD, key, feature_types,
                                                    dense_types=("ADT",))
    results[key] = dict(summary=summary, per_cell=per_cell, adt=dense["ADT"].astype(np.float64))
    print(f"=== {key}")
    display(summary.T)

=== layers/counts


feature_type,GEX,ADT
n_features,1.395e+04,134
n_stored,1.314e+08,1.014e+07
n_negative,0,0
n_non_integer,0,0
n_explicit_zero,0,0
min,1,1
max,3.545e+04,3.245e+04
density,0.1043,0.8385
frac_non_integer,0,0


=== X


feature_type,GEX,ADT
n_features,1.395e+04,134
n_stored,1.314e+08,1.014e+07
n_negative,0,0
n_non_integer,1.314e+08,1.014e+07
n_explicit_zero,0,0
min,0.1043,0.01248
max,2.108e+07,7.911
density,0.1043,0.8385
frac_non_integer,1,1


`layers['counts']`: all stored values are positive integers in both GEX and ADT, so
it holds **raw counts**. `X`: no negative values, almost no integers, and the
same number of stored entries as `counts`.

## 2. GEX: is `X = counts / size_factor`?

If so, within one cell the ratio `X / counts` is the same for every gene and
equals `1 / size_factor`.

In [3]:
gex_cols = np.flatnonzero(feature_types == "GEX")
ratio = per_cell_ratio(CITE_H5AD, "X", "layers/counts", gex_cols)
print("same sparsity pattern in every cell:", ratio.same_pattern.all())
spread = (ratio.ratio_max - ratio.ratio_min) / ratio.ratio_median
print("max relative spread of X/counts within a cell:", np.nanmax(spread))
print("max |median(X/counts) * size_factor - 1|:", np.nanmax(np.abs(ratio.ratio_median * size_factors - 1)))

same sparsity pattern in every cell: True
max relative spread of X/counts within a cell: 2.1237472e-07
max |median(X/counts) * size_factor - 1|: 5.9351718872946435e-08


## 3. Size factors look suspicious

scran size factors are usually centred around 1 and track library size.

In [4]:
lib_size = results["layers/counts"]["per_cell"]["GEX"]["sum"]
sf = pd.Series(size_factors)
display(sf.describe(percentiles=[.001, .01, .05, .5, .95, .99]))
print("cells with size factor < 0.1:", (sf < 0.1).sum(), f"({(sf < 0.1).mean():.1%})")
print("Spearman(size factor, library size):", sf.corr(lib_size, method="spearman"))
pd.DataFrame({"size_factor": sf, "library_size": lib_size}).nsmallest(5, "size_factor")

count   9.026e+04
mean       0.9479
std        0.8587
min     0.0001965
0.1%     0.008554
1%        0.02221
5%        0.07486
50%        0.7204
95%         2.555
99%         4.806
max         9.589
dtype: float64

cells with size factor < 0.1: 4874 (5.4%)


Spearman(size factor, library size): 0.5110637208415755


,size_factor,library_size
35473,0.0001965,5160
41028,0.002409,4264
28376,0.002663,3666
36973,0.003108,3709
31190,0.004017,5187


## 4. ADT: which CLR?

A common CLR variant (Seurat-style) is `X = log1p(counts / g)`. Then
`g = counts / expm1(X)` on non-zero entries. If `g` is constant within each cell,
CLR was computed per cell (across proteins); if constant within each protein,
per protein (across cells).

In [5]:
C = results["layers/counts"]["adt"]
X = results["X"]["adt"]
nz = C > 0
print("zeros identical in X and counts:", np.array_equal(C == 0, X == 0))
G = np.full_like(C, np.nan)
G[nz] = C[nz] / np.expm1(X[nz])

def rel_spread(A, axis):
    return (np.nanmax(A, axis) - np.nanmin(A, axis)) / np.nanmedian(A, axis)

print("relative spread of g within each cell   : max %.1e" % np.nanmax(rel_spread(G, 1)))
print("relative spread of g within each protein: median %.1f" % np.nanmedian(rel_spread(G, 0)))
g_cell = np.nanmedian(G, axis=1)

zeros identical in X and counts: True


relative spread of g within each cell   : max 4.8e-07


relative spread of g within each protein: median 17.5


## 5. What goes into the per-cell denominator `g`?

The isotype controls in `obsm` are not integers, so they were transformed too.
Inverting with the same per-cell `g` should give back integer raw counts.

In [6]:
with h5py.File(CITE_H5AD) as f:
    grp = f["obsm/ADT_isotype_controls"]
    iso_names = [c.decode() if isinstance(c, bytes) else c for c in grp.attrs["column-order"]]
    ISO = np.column_stack([read_df_column(grp, c) for c in iso_names]).astype(np.float64)
    iso_count = read_df_column(f["obs"], "ADT_iso_count")
    adt_total = read_df_column(f["obs"], "ADT_total_counts")

iso_raw = np.expm1(ISO) * g_cell[:, None]
print("recovered isotype counts, max distance to an integer:", np.abs(iso_raw - iso_raw.round()).max())
iso_raw = iso_raw.round()
print("obs ADT_iso_count == recovered isotype sum:", np.allclose(iso_count, iso_raw.sum(1)))
print("obs ADT_total_counts == 134 ADT + 6 isotype counts:", np.allclose(adt_total, C.sum(1) + iso_raw.sum(1)))

g140 = np.exp(np.log1p(np.hstack([C, iso_raw])).mean(axis=1))
print("max |X_ADT - log1p(counts / g140)|:", np.abs(np.log1p(C / g140[:, None]) - X).max())

recovered isotype counts, max distance to an integer: 6.611885368101866e-06
obs ADT_iso_count == recovered isotype sum: True
obs ADT_total_counts == 134 ADT + 6 isotype counts: True


max |X_ADT - log1p(counts / g140)|: 2.3839992557839196e-07


## Findings

| Block | `layers['counts']` | `X` | Status |
|---|---|---|---|
| GEX (13,953) | raw UMI counts | `counts / obs['GEX_size_factors']`, **no log** | verified (rel. error ~1e-7) |
| ADT (134) | raw ADT counts | per-cell CLR: `log1p(x / exp(mean_j log1p(x_j)))`, mean over **140** features (134 + 6 isotypes) | verified (abs. error ~2e-7) |
| `obsm['ADT_isotype_controls']` | — | same CLR, not raw counts | verified |

* Some GEX size factors are implausibly small (min ~0.0002 for a cell with ~5,000 UMIs),
  so `X` for GEX has extreme values. Why is not verified. **Decision:** normalize GEX from
  `layers['counts']` ourselves in Phase 1.
* Our tests identify the formula, not the software that produced it.